In [ ]:
!pip install pytesseract
!pip install pillow
!pip install opencv-python

In [ ]:
# ocr  pyterract
from PIL import Image
import pytesseract

In [ ]:
img = Image.open("/content/passbook.png")
ocr = pytesseract.image_to_string(img)
print(ocr)
print(type(ocr))

NATIONAL UNION BANK

Savings Passbook | A/C No: 3021 4456 7890 | Page 4

 

 

Date Particulars Withdrawal Deposit Balance
61/07/2625 — UPI-SWIGGY-ORDER-8827 450.00 12,340.00
62/07/2025 SALARY CREDIT JULY 55,000.00 67,340.00
03/07/2025 — UPI-RENT-LANDLORD-XYZ 15,000.00 52,340.00
05/07/2025  UPI-BIGBASKET-GROCERY 2,340.00 50,000.00
06/07/2025 — NEFT-ELECTRICITY-BOARD 1,200.00 48,800.00
08/07/2025 — UPI-ZOMATO-ORDER-991 680.00 48,120.00
10/07/2025  ATM-WITHDRAWAL-CASH 5,000.00 43,120.00
12/07/2025 UPI-NETFLIX-SUBSCRIPTION 499.00 42,621.00
15/07/2025  UPI-BIGBASKET-GROCERY 2,100.00 40,521.00
18/07/2025 | MUTUAL-FUND-SIP-DEBIT 5,000.00 35,521.00
20/07/2625 — UPI-SWIGGY-ORDER-9012 520.00 35,001.00
22/07/2025 UPI-UBER-TRIP 340.00 34,661.00
25/07/2025 — UPI-AMAZON-SHOPPING 3,200.00 31,461.00
28/07/2025 — UPI-ZOMATO-ORDER-1204 610.00 30,851.00
30/07/2025 — NEFT-ELECTRICITY-BOARD 1,150.00 29,701.00

<class 'str'>


In [ ]:
import pandas as pd

In [ ]:
#structure to the text (regex)
import re
Lin_re = re.compile(
    r"(?P<date>\d{2}/\d{2}/\d{4})\s*[—\-\|]*\s"
    r"(?P<desc>.+?)\s+"
    r"(?P<amount>[\d,]+\.\d{2})\s+"
    r"(?P<balance>[\d,]+\.\d{2}\s*$)"
)

def am_type(desc):
  if re.search(r'salary|credit|refund',desc, re.I):
    return "CR"
  else:
    return "DR"

rows = []

text = ocr.strip().splitlines()
for i in text:
  reg = Lin_re.search(i.strip())
  if not reg:
    continue
  rows.append({
      "date":reg.group("date"),
      "desc":reg.group("desc"),
      "type":am_type(reg.group("desc")),
      "amount":float(reg.group("amount").replace(",","")),
      "balance":float(reg.group("balance").replace(',',''))
  })

df = pd.DataFrame(rows)
df

,date,desc,type,amount,balance
0,61/07/2625,UPI-SWIGGY-ORDER-8827,DR,450.0,12340.0
1,62/07/2025,SALARY CREDIT JULY,CR,55000.0,67340.0
2,03/07/2025,UPI-RENT-LANDLORD-XYZ,DR,15000.0,52340.0
3,05/07/2025,UPI-BIGBASKET-GROCERY,DR,2340.0,50000.0
4,06/07/2025,NEFT-ELECTRICITY-BOARD,DR,1200.0,48800.0
5,08/07/2025,UPI-ZOMATO-ORDER-991,DR,680.0,48120.0
6,10/07/2025,ATM-WITHDRAWAL-CASH,DR,5000.0,43120.0
7,12/07/2025,UPI-NETFLIX-SUBSCRIPTION,DR,499.0,42621.0
8,15/07/2025,UPI-BIGBASKET-GROCERY,DR,2100.0,40521.0
9,18/07/2025,MUTUAL-FUND-SIP-DEBIT,DR,5000.0,35521.0


In [ ]:
from transformers import pipeline

In [ ]:
zshot = pipeline('zero-shot-classification', model = 'facebook/bart-large-mnli')

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
category_labels = [
    "groceries","rent","electricity or utility bill", "food delivery or dining out",
    "entertainment subscription","shopping","cash withdrawal","salary","investment or savings"
]

In [ ]:

preds = zshot(list(df['desc']), candidate_labels = category_labels, batch_size =8)
category = []
for i in preds:
  category.append(i['labels'][0])
df['category'] = category
df

,date,desc,type,amount,balance,category
0,61/07/2625,UPI-SWIGGY-ORDER-8827,DR,450.0,12340.0,electricity or utility bill
1,62/07/2025,SALARY CREDIT JULY,CR,55000.0,67340.0,salary
2,03/07/2025,UPI-RENT-LANDLORD-XYZ,DR,15000.0,52340.0,rent
3,05/07/2025,UPI-BIGBASKET-GROCERY,DR,2340.0,50000.0,groceries
4,06/07/2025,NEFT-ELECTRICITY-BOARD,DR,1200.0,48800.0,electricity or utility bill
5,08/07/2025,UPI-ZOMATO-ORDER-991,DR,680.0,48120.0,food delivery or dining out
6,10/07/2025,ATM-WITHDRAWAL-CASH,DR,5000.0,43120.0,cash withdrawal
7,12/07/2025,UPI-NETFLIX-SUBSCRIPTION,DR,499.0,42621.0,entertainment subscription
8,15/07/2025,UPI-BIGBASKET-GROCERY,DR,2100.0,40521.0,groceries
9,18/07/2025,MUTUAL-FUND-SIP-DEBIT,DR,5000.0,35521.0,investment or savings


In [ ]:
def analyze(df):
  debits = df[df['type']== "DR"]
  credits = df[df['type']== "CR"]

  total_spend = debits["amount"].sum()
  total_income = credits['amount'].sum()

  by_category = (debits.groupby('category')['amount'].sum().sort_values(ascending = False))

  savings_rate = (total_income - total_spend)/total_income

  return {
      "total_spend":total_spend,
      "total_income":total_income,
      "by_category":by_category,
      "savings_rate":savings_rate
  }

stats = analyze(df)

print(str(stats))

{'total_spend': np.float64(38089.0), 'total_income': np.float64(55000.0), 'by_category': category
rent                           15000.0
cash withdrawal                 5000.0
investment or savings           5000.0
groceries                       4440.0
electricity or utility bill     3320.0
shopping                        3200.0
food delivery or dining out     1290.0
entertainment subscription       839.0
Name: amount, dtype: float64, 'savings_rate': np.float64(0.3074727272727273)}


In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer


model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


...

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

In [ ]:
import torch

In [ ]:
model = model.to("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
print(stats["by_category"])

category
rent                           15000.0
cash withdrawal                 5000.0
investment or savings           5000.0
groceries                       4440.0
electricity or utility bill     3320.0
shopping                        3200.0
food delivery or dining out     1290.0
entertainment subscription       839.0
Name: amount, dtype: float64


In [ ]:
def build_briefing(stats):
    cat_lines = "\n".join(
        f"- {cat}: \u20b9{amt:.0f}" for cat, amt in stats["by_category"].items()
    )
    return (
        f"Monthly income: \u20b9{stats['total_income']:.0f}\n"
        f"Monthly spend: \u20b9{stats['total_spend']:.0f}\n"
        f"Savings rate: {stats['savings_rate']*100:.1f}%\n"
        f"Spending by category:\n{cat_lines}\n"
    )

stats = analyze(df)

print("Income:  \u20b9%.2f" % stats["total_income"])
print("Spend:   \u20b9%.2f" % stats["total_spend"])
print("Savings rate: %.1f%%" % (stats["savings_rate"] * 100))
print()
print("Spend by category:")
print(stats["by_category"])
print()
print("Recurring payments:")
print(stats["recurring"])

Income:  ₹55000.00
Spend:   ₹38089.00
Savings rate: 30.7%

Spend by category:
category
rent                           15000.0
cash withdrawal                 5000.0
investment or savings           5000.0
groceries                       4440.0
electricity or utility bill     3320.0
shopping                        3200.0
food delivery or dining out     1290.0
entertainment subscription       839.0
Name: amount, dtype: float64

Recurring payments:


KeyError: 'recurring'

In [ ]:
def generate_advice(stats):
  stats = build_briefing(stats)
  prompt = (
      "You are a personal finance assistant. based on the finace stats, help this person save effeciently, this is the finance data of user: "+ocr
  )
  inputs = tokenizer(prompt,return_tensors="pt", max_length = 512).to(model.device)
  gen = model.generate(**inputs, max_new_tokens = 200)
  return tokenizer.decode(gen[0])

print(generate_advice(stats))